# Dropout

_Deep Learning (CS230)_

**Randomly switch off neurons during training so the network can't lean on any one of them.**

Overfitting means the network memorizes the training data instead of learning general patterns.

     Dropout fights this. During training, it randomly turns off some neurons each step.

---

This notebook is a practice scaffold for the **Dropout** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np, matplotlib.pyplot as plt

## Reference implementation — PyTorch

In [ ]:
import torch
import torch.nn as nn

net = nn.Sequential(
    nn.Linear(20, 64), nn.ReLU(),
    nn.Dropout(p=0.5),        # drop 50% of neurons each forward pass
    nn.Linear(64, 1),
)

x = torch.randn(4, 20)

net.train()                   # training mode: dropout ON (random zeros)
print("train out:", net(x).squeeze().round(decimals=3).tolist())

net.eval()                    # eval mode: dropout OFF (use full network)
with torch.no_grad():
    print("eval  out:", net(x).squeeze().round(decimals=3).tolist())

## Visualize the data & results

_Training on real tumor data, does strong regularization (dropout-style) close the train-validation gap?_

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import log_loss

bc = load_breast_cancer()
X = StandardScaler().fit_transform(bc.data)
Xtr, Xte, ytr, yte = train_test_split(X, bc.target, test_size=0.3,
                                      random_state=0, stratify=bc.target)

def curves(alpha):
    m = MLPClassifier(hidden_layer_sizes=(64, 64), solver="adam", max_iter=1,
                      warm_start=True, random_state=0, alpha=alpha,
                      learning_rate_init=0.003)
    tr, va = [], []
    for e in range(40):
        m.partial_fit(Xtr, ytr, classes=[0, 1]) if e == 0 else m.partial_fit(Xtr, ytr)
        tr.append(log_loss(ytr, m.predict_proba(Xtr), labels=[0, 1]))
        va.append(log_loss(yte, m.predict_proba(Xte), labels=[0, 1]))
    return tr, va

tr_w, va_w = curves(1e-5)               # weak reg, overfits
_, va_s = curves(3.0)                   # strong reg (dropout-like)

plt.plot(tr_w, color="#4ea1ff", label="train (weak reg)")
plt.plot(va_w, color="#ff7b72", label="val (weak reg, overfits)")
plt.plot(va_s, color="#7ee787", label="val (strong reg)")
plt.title("Real train vs validation log-loss on breast-cancer data")
plt.xlabel("epoch"); plt.ylabel("log loss")
plt.legend()
plt.show()